<a id="unit-u.09735117d2f74abd9d89af963625aab4"></a>

<div>
  <div>05-ЗЕРТХАНАЛЫҚ ЖҰМЫС · APACHE DORIS КӨМЕГІМЕН НАҚТЫ УАҚЫТТАҒЫ АНАЛИТИКА</div>
  <div>Doris функцияларын зерттеу және талдамалық сұраулар құру</div>
  <p>Функцияны бизнес сұрағына қажетті нәтиже пішіні мен тәуелділікке қарай таңдап, содан кейін егжей-тегжейлі оқиға жолдарын топталған метрикаларға, сүзілген топтарға, күн-аймақ үрдістеріне және толық талдамалық нәтижеге айналдырыңыз.</p>
  <span>Скалярлық · Агрегаттық · Комбинатор · Терезелік · Table Function · TVF · AI · CTE · Lambda · UDF тобы</span>
</div>

**5-модуль: Apache Doris жүйесінде деректерді талдау** · [Сабақ](course.md) · [Тест](quiz05_analytical_query_semantics.ipynb)

Шамамен 30 минут жоспарлаңыз.

Бұл зертханалық жұмыс 4-модульде жасалған толық `events_modelled` кестесімен жалғасады. Ноутбук орындалатын SQL, шағын литерал мысалдар және интерактивті әрекеттер береді, сондықтан әр функцияның кіріс және шығыс жолдарына не істейтініне көңіл бөле аласыз. Зертханалық жұмыс сыртқы AI жеткізушісіне хабарласпайды және пайдаланушы кодын орналастырмайды.

<a id="unit-u.6d16b1bafd794eb2ba1361427b22ba8d"></a>

<a id="initialize-the-lab"></a>

### Зертханалық жұмысты бастапқы баптау

Келесі ұяшықты 1-бөлімге дейін орындаңыз. Ол ортақ курс көмекшісін жүктеп, кейінгі әр ұяшық пайдаланатын `lab` нысанын жасайды. Jupyter ядросын қайта іске қосқаннан кейін оны қайта орындаңыз. Ол Docker іске қоспайды және Doris деректерін өзгертпейді.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT);


<a id="unit-u.4aa31a6cb0264c6ca8bc0676b7bbab3a"></a>

<a id="1-choose-a-function-category-from-the-required-result-shape"></a>

## 1. Қажетті нәтиже пішініне қарай функция санатын таңдау

Doris функцияларды орындайтын жұмыс түріне қарай топтайды. Каталогты жаттаудың орнына кіріс, шығыс, жол саны әрекеті және сыртқы тәуелділіктерден бастаңыз.

| Функция санаты | Кіріс және шығыс әрекеті | Өкілдік мысал |
|---|---|---|
| Скалярлық функция | Әр кіріс жолына бір мән береді | `LOWER(event_type)` |
| Агрегат функциясы | Бірнеше жол мәндерін әр топқа бір мәнге біріктіреді | `SUM(revenue)` |
| Комбинатор | Агрегат әрекетін арнайы күй немесе ARRAY міндетіне бейімдейді | `SUM_FOREACH(metric_array)` |
| Талдамалық (терезелік) функция | Кіріс нәтиже жолдарын сақтай отырып есептелген мән қосады | `ROW_NUMBER() OVER (...)` |
| Table Function | Әр ағымдағы кіріс жолының мәнін нөлден көпке дейін жолға жаяды | `LATERAL VIEW` арқылы `EXPLODE(tags)` |
| Table-Valued Function (TVF) | `FROM` ішінде бола алатын қатынас береді | `NUMBERS(...)` немесе `S3(...)` |
| AI функциясы | Бапталған Doris AI Resource арқылы модель шақырады | `AI_SENTIMENT(resource, text)` |

Бұл санаттар бір қырында қабаттасуы мүмкін. Мысалы, `AI_SENTIMENT` скалярлық жол әрекетіне ие, ал AI санаты шақырудың сыртқы модельге тәуелді екенін де айтады. `OVER` арқылы қолданылған агрегат терезе мағынасына сай болып, кіріс нәтиже жолдарын сақтайды.

Әр өрнекті талабына сәйкестендіріңіз. AI мысалы мұнда жіктеледі, бірақ зертханалық жұмыста орындалмайды, өйткені оған AI Resource, жеткізушіге қатынау, тіркелгі деректері және сыртқы модель шақыруы қажет.

In [ ]:
lab.function_category_activity();


<a id="unit-u.5e0d3a1a2c44427189d533ed357ba4ae"></a>

<a id="2-distinguish-a-table-function-from-a-tvf-by-where-its-rows-come-from"></a>

## 2. Table Function мен TVF жолдарының қайдан келетіні арқылы ажырату

Екі санат та бірнеше жол қайтара алады, бірақ сұрауға әртүрлі кіреді.

- **Table Function** әр ағымдағы кіріс жолынан мән жаяды. `EXPLODE(tags)` `LATERAL VIEW` арқылы қолданылады, сондықтан жасалған жолдар `source_id` мәнімен байланысты қалады.
- **Table-Valued Function (TVF)** — `FROM` ішіндегі қатынастың өзі. `NUMBERS` бастапқы кестесіз жол жасайды.

Келесі ұяшық алдымен сынақ ортасы тоқтап тұрса оны іске қосып, кейінгі бөлімдер оқитын 4-зертханалық жұмыстың `events_modelled` кестесі толық екенін тексереді. Мысалдардың өзі литералдар мен жасалған сандар пайдаланады, сондықтан объектілік сақтау не сыртқы қызмет қажет емес. 3-модульдің `S3` функциясы сол TVF қатынас шартына сай, бірақ бұл жұмыс S3 қайта оқымайды.

In [ ]:
lab.start_container()
lab.connect()
lab.execute("USE doris_course")
lab.check_prerequisites(5)

lab.sql("""
SELECT
    source_id,
    tag
FROM (
    SELECT 1 AS source_id, ['view', 'purchase'] AS tags
) seed
LATERAL VIEW EXPLODE(tags) expanded AS tag
ORDER BY tag
""", title="Table Function: expand one input row")

lab.sql("""
SELECT number
FROM NUMBERS("number" = "4")
ORDER BY number
""", title="TVF: generate a temporary relation");

<a id="unit-u.48b661c73d9b44f385f0eb2c3db273df"></a>

**Күтілетін нәтиже:** жасыл белгі `events_modelled` ішінде 10,158,080 жол барын растайды (әйтпесе ұяшық тоқтап, 4-зертханалық жұмысты атайды). `EXPLODE` сұрауы бір `seed` жолын `(1, purchase)` және `(1, view)` екі жолына айналдырады. Екі жол да `source_id = 1` сақтайды, бұл сол кіріс жолынан шыққанын көрсетеді. Кіріс кестесі болмаса да, `NUMBERS` сұрауы төрт жол — `0`, `1`, `2` және `3` қайтарады.

| Амал | Бар кіріс жолдары | Жасалған нәтиже жолдары |
|---|---:|---:|
| `EXPLODE(tags)` | Екі элементті ARRAY қамтитын 1 жол | Сол кіріс жолына байланған 2 жол |
| `NUMBERS("number" = "4")` | Жоқ | Қатынас ретінде берілген 4 жол |

<a id="unit-u.9735c58a37554154a524202985ab5424"></a>

<a id="3-turn-stored-values-into-reporting-dimensions"></a>

## 3. Сақталған мәндерді есеп өлшемдеріне айналдыру

2020-03-03 күнінен сәйкес сегіз оқиғаны тексеріңіз. Кесте бастапқы `event_time` және `region` сақтайды; сұрау орындалғанда скалярлық функциялар есеп мәндерін шығарады:

- `TO_DATE` уақыт белгісін күнтізбелік күніне түрлендіреді.
- `DATE_TRUNC(..., 'hour')` уақыт белгілерін сағаттық шекараға қалыпқа келтіреді.
- `EXTRACT` бір уақыт құрамдасын қайтарады.
- `REPLACE` және `UPPER` көрсетілім белгісін жасайды.
- `LIKE` және `REGEXP` жол үлгісімен жолдарды таңдайды.

Әр таңдалған кіріс оқиғасы әлі бір шығыс жолын береді, сондықтан бұл скалярлық функциялар нәтиже түйіршіктілігін өзгертпейді.

In [ ]:
lab.sql("""
SELECT
    event_time,
    TO_DATE(event_time) AS event_date,
    DATE_TRUNC(event_time, 'hour') AS event_hour,
    EXTRACT(HOUR FROM event_time) AS hour_of_day,
    event_type,
    UPPER(REPLACE(region, 'region_', 'REGION-')) AS region_label
FROM events_modelled
WHERE event_time >= '2020-03-03 00:00:00'
  AND event_time <  '2020-03-04 00:00:00'
  AND region LIKE 'region_0%'
  AND event_type REGEXP '^(view|purchase)$'
ORDER BY event_time, event_id
LIMIT 8
""", title="Scalar-function result");


<a id="unit-u.6aba402c5e644d019ed8d64db2fd7d0d"></a>

**Күтілетін нәтиже:** сегіз жол қайтарылады. Бірінші жолда `event_date = 2020-03-03`, `event_hour = 2020-03-03 00:00:00`, `hour_of_day = 0` және `region_label = REGION-01` бар. Сақталған мәндердің туынды мәндер жанында қолжетімді қалатынына назар аударыңыз.


<a id="unit-u.fce2b898abc44176b551bbad839a6177"></a>

<a id="4-reduce-detail-rows-to-grouped-business-metrics"></a>

## 4. Егжей-тегжейлі жолдарды топталған бизнес метрикаларына азайту

Келесі сұрау түйіршіктілікті әр бастапқы оқиғаға бір жолдан әр күн мен аймаққа бір жолға өзгертеді. `COUNT`, `MIN`, `MAX` және `SUM` — көптеген кіріс жол мәндерін біріктіретіндіктен агрегат функциялары. Таңдалған күн үшін 140,782 оқиға жолы сегіз топталған жолға айналады.

Сатып алу есептері әр агрегат функциясының ішіне шартты өрнек қояды. Сатып алу емес жолдар `event_count` мәніне бәрібір үлес қосады, бірақ сатып алу `MIN` және `MAX` үшін `NULL`, ал `purchase_revenue` үшін нөл береді. Осылайша жалпы белсенділік пен сатып алуға тән метрикалар сол топталған жолда беріледі.


In [ ]:
lab.sql("""
SELECT
    TO_DATE(event_time) AS event_date,
    region,
    COUNT(*) AS event_count,
    MIN(CASE WHEN event_type = 'purchase' THEN revenue END) AS min_purchase_value,
    MAX(CASE WHEN event_type = 'purchase' THEN revenue END) AS max_purchase_value,
    SUM(CASE WHEN event_type = 'purchase' THEN revenue ELSE 0 END) AS purchase_revenue
FROM events_modelled
WHERE event_time >= '2020-03-03 00:00:00'
  AND event_time <  '2020-03-04 00:00:00'
GROUP BY TO_DATE(event_time), region
ORDER BY purchase_revenue DESC, region
""", title="Daily metrics by region");


<a id="unit-u.dc5d9ceda8a542dca1d41c72035506d0"></a>

**Күтілетін нәтиже:** сегіз топталған жол қайтарылады: `2020-03-03` күнінде әр аймаққа біреу. Сегіз `event_count` мәнін қосқанда **140,782** кіріс оқиға, ал `purchase_revenue` қосқанда **6,246,556.39** шығады. `min_purchase_value` және `max_purchase_value` тек сатып алу жолдарын сипаттайды, өйткені агрегат функциялары басқа оқиға түрлері үшін алынған `NULL` елемейді.


<a id="unit-u.7601418dfa3f4e1fad6a7136a3164b58"></a>

<a id="5-filter-input-rows-with-where-and-aggregated-groups-with-having"></a>

## 5. Кіріс жолдарын WHERE, агрегатталған топтарды HAVING арқылы сүзу

Бұл сұрау бір күнде кемінде 120,000 сатып алу түсімін жасаған өнімдерді сұрайды.

| Тармақ | Немен жұмыс істейді | Осы сұраудағы мағынасы |
|---|---|---|
| `WHERE` | Бастапқы оқиға жолдары | Агрегаттауға тек таңдалған күннің сатып алу оқиғалары кіреді |
| `GROUP BY` | Сүзілген кіріс жолдары | Әр `product_id` үшін бір топ жасалады |
| `HAVING` | Агрегатталған топтар | Кемінде 120,000 түсімі бар өнім топтары ғана қалады |

Түсім шегін `WHERE` ішіне көшіру басқа сұрақ қояр еді: өнім жиынтықтарының орнына жеке сатып алуларды сүзер еді.


In [ ]:
lab.sql("""
SELECT
    product_id,
    COUNT(*) AS purchase_events,
    SUM(revenue) AS purchase_revenue
FROM events_modelled
WHERE event_time >= '2020-03-03 00:00:00'
  AND event_time <  '2020-03-04 00:00:00'
  AND event_type = 'purchase'
GROUP BY product_id
HAVING SUM(revenue) >= 120000
ORDER BY purchase_revenue DESC, product_id
""", title="Product groups above the revenue threshold");


<a id="unit-u.ca91e18337e04ede874795ba9bcae359"></a>

**Күтілетін нәтиже:** `HAVING` кейін алты өнім тобы қалады. `1005115` өнімі 593 сатып алу оқиғасы және 516,793.33 түсіммен бірінші; `1004249` өнімі 126,135.40 түсіммен соңғы жарамды топ.


<a id="unit-u.0861950b0be346f2b54bdcaec88798cb"></a>

<a id="6-use-any_value-only-when-any-representative-value-is-valid"></a>

## 6. ANY_VALUE тек кез келген өкілдік мән жарамды болғанда пайдалану

Топталған сұрау бағанды тек ол топты анықтаса немесе агрегат функциясымен есептелсе ғана көрсете алады. Бірінші сұрау бұл ережені әдейі бұзады: `region` бойынша топтайды, бірақ агрегатталмаған `event_time` мәнін де сұрайды. Doris уақыт белгісін үнсіз таңдаудың орнына екіұшты проекцияны қабылдамайды.


In [ ]:
lab.expected_sql_error("""
SELECT
    region,
    TO_DATE(event_time) AS event_date,
    COUNT(*) AS event_count
FROM events_modelled
WHERE event_time >= '2020-03-03 00:00:00'
  AND event_time <  '2020-03-04 00:00:00'
GROUP BY region
ORDER BY region
""", contains="not in aggregate", title="Doris rejects an ambiguous grouped projection");


<a id="unit-u.2fbd8b24fe43451a858676b877edc426"></a>

Күн предикаты әр таңдалған оқиғаның бір күнтізбелік күнге жататынын кепілдейді. Дәл осы шартта әр аймақтан кез келген `NULL` емес күнді таңдау мағыналы, сондықтан `ANY_VALUE` ниетті айқын етеді. Оны бөлек топтарды анықтауы тиіс өлшемді жасыру үшін пайдаланбаңыз.


In [ ]:
lab.sql("""
SELECT
    region,
    ANY_VALUE(TO_DATE(event_time)) AS event_date,
    COUNT(*) AS event_count
FROM events_modelled
WHERE event_time >= '2020-03-03 00:00:00'
  AND event_time <  '2020-03-04 00:00:00'
GROUP BY region
ORDER BY region
""", title="A valid representative date for each region");


<a id="unit-u.1c2257762a554e1aa3de521547329ce7"></a>

**Күтілетін нәтиже:** сегіз аймақ жолы қайтарылады және әр `event_date` — `2020-03-03`. Олардың `event_count` мәндері әлі 140,782 болып қосылады. `ANY_VALUE` мұнда жарамды, өйткені күн предикаты өкілдік күнді барлық ықтимал кіріс мәндерінде бірдей етеді.


<a id="unit-u.3c0839c04b9b41a98f0163488e804e49"></a>

<a id="7-name-an-aggregated-intermediate-result-with-a-cte"></a>

## 7. Агрегатталған аралық нәтижеге CTE арқылы атау беру

Common Table Expression (CTE) бір оператор ішіндегі аралық нәтижеге атау береді. Мұнда `daily` егжей-тегжейлі оқиға жолдары әр күнге бір жолға агрегатталғаннан кейін алынған нәтижені атайды. Ол тұрақты кесте жасамайды.

Келесі бөлімге сегіз жолдық кіріс көрінуі үшін бұл бірінші сұрау агрегаттаудан кейін әдейі тоқтайды.


In [ ]:
lab.sql("""
WITH daily AS (
    SELECT
        TO_DATE(event_time) AS event_date,
        SUM(CASE WHEN event_type = 'purchase' THEN revenue ELSE 0 END) AS daily_revenue
    FROM events_modelled
    WHERE event_time >= '2020-03-01 00:00:00'
      AND event_time <  '2020-03-09 00:00:00'
    GROUP BY TO_DATE(event_time)
)
SELECT
    event_date,
    daily_revenue
FROM daily
ORDER BY event_date
""", title="Daily aggregate produced by a CTE");


<a id="unit-u.c80aeef87834435099704682288a7ab0"></a>

**Күтілетін нәтиже:** агрегаттау сегіз күндік жол береді. `2020-03-01` күнінде түсім 5,670,241.29, `2020-03-02` күнінде 9,933,097.98 және `2020-03-03` күнінде 6,246,556.39. `2020-03-04` пен `2020-03-08` арасындағы бес күн 0.00 көрсетеді: 1-модульде сипатталғандай, курс деректерінде ол күндері қараулар және `2020-03-08` күнінде себет оқиғалары бар, бірақ сатып алу жоқ. Оқиғалары бар болғандықтан, күндер бәрібір көрінеді; мүлде оқиғасы жоқ күннің жолы болмас еді. Бұл сегіз жол келесі бөлімдегі терезелік функцияларға кіріс нәтиже жолдары болады.

<a id="unit-u.f7884d31dba7476caf4aef34fece44f5"></a>

<a id="8-add-window-calculations-while-retaining-the-daily-rows"></a>

## 8. Күндік жолдарды сақтай отырып терезелік есептеулер қосу

Сұрау сол `daily` CTE арқылы, демек сол сегіз күндік жолдан басталады. Терезелік функциялар содан кейін әр жолға есептеулер қосады:

- `LAG` алдыңғы күндік нәтижені оқиды.
- `SUM(...) OVER (...)` жинақталған қосынды береді.
- `ROW_NUMBER` детерминді түсім орнын тағайындайды.

Бұрынғы агрегат функцияларынан айырмашылығы, бұл терезелік функциялар сегіз кіріс нәтиже жолын біріктіріп азайтпайды.


In [ ]:
lab.sql("""
WITH daily AS (
    SELECT
        TO_DATE(event_time) AS event_date,
        SUM(CASE WHEN event_type = 'purchase' THEN revenue ELSE 0 END) AS daily_revenue
    FROM events_modelled
    WHERE event_time >= '2020-03-01 00:00:00'
      AND event_time <  '2020-03-09 00:00:00'
    GROUP BY TO_DATE(event_time)
)
SELECT
    event_date,
    daily_revenue,
    LAG(daily_revenue, 1, 0) OVER (ORDER BY event_date) AS previous_revenue,
    SUM(daily_revenue) OVER (
        ORDER BY event_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_revenue,
    ROW_NUMBER() OVER (
        ORDER BY daily_revenue DESC, event_date
    ) AS revenue_rank
FROM daily
ORDER BY event_date
""", title="Window calculations over the daily rows");


<a id="unit-u.b7dfc385b7824003b0cca0dbedbba098"></a>

**Күтілетін нәтиже:** нәтиже `daily` CTE жасаған сол сегіз күнді қамтиды, бірақ енді әр жолда алдыңғы түсім, жинақталған түсім және орын мәндері бар. `2020-03-02` күнінде алдыңғы түсім 5,670,241.29, жинақталған түсім 15,603,339.27.

| Сұрау кезеңі | Кіріс жолдары | Шығыс жолдары | Жол әрекеті |
|---|---:|---:|---|
| Күндік агрегаттау | Егжей-тегжейлі оқиға жолдары | 8 | Жолдар күндік түйіршіктілікке біріктіріледі |
| Терезелік есептеу | 8 күндік жол | 8 | Күндік жолдар сақталып, есептелген бағандар қосылады |


<a id="unit-u.1a1ef1896e5742ce8890fafd17f099fc"></a>

<a id="9-combine-the-stages-into-a-daily-region-business-result"></a>

## 9. Кезеңдерді күн-аймақ бизнес нәтижесіне біріктіру

Операциялық топқа 1–3 наурыз үшін әр күн мен аймаққа бір жол қажет. Әр жол барлық оқиға белсенділігін, бірегей белсенді пайдаланушыларды, бірегей сатып алушыларды, сатып алу түсімін, сол аймақтың алдыңғы күн түсімін, жинақталған аймақтық түсімді және күн ішіндегі аймақтың түсім орнын көрсетуі тиіс.

Сұрау осы міндеттерді бөледі:

1. `filtered` кіріс жиынын таңдап, `event_date` шығарады. Оның түйіршіктілігі әр оқиғаға бір жол болып қалады.
2. `daily_region` түйіршіктілікті әр `(event_date, region)` үшін бір жолға өзгертіп, сол топтың метрикаларын есептейді.
3. `scored` 24 күн-аймақ жолына терезелік функциялар қолданады. `PARTITION BY region` әр аймақтың уақыт қатарын анықтайды; `PARTITION BY event_date` әр күндегі орын жарысын анықтайды.
4. Сыртқы сұрау соңғы жолдарды көрсету үшін реттейді.

Common Table Expression (CTE) осы оператор ішінде нәтиже кезеңіне атау береді. Ол тұрақты кесте жасамайды және оның жай болуы материалдандыруды немесе өнімділік жақсаруын уәде етпейді.

In [ ]:
lab.sql("""
WITH filtered AS (
    SELECT
        TO_DATE(event_time) AS event_date,
        region,
        user_id,
        event_type,
        revenue
    FROM events_modelled
    WHERE event_time >= '2020-03-01 00:00:00'
      AND event_time <  '2020-03-04 00:00:00'
),
daily_region AS (
    SELECT
        event_date,
        region,
        COUNT(*) AS event_count,
        COUNT(DISTINCT user_id) AS active_users,
        COUNT(DISTINCT CASE
            WHEN event_type = 'purchase' THEN user_id
        END) AS purchase_users,
        SUM(CASE
            WHEN event_type = 'purchase' THEN revenue ELSE 0
        END) AS purchase_revenue
    FROM filtered
    GROUP BY event_date, region
),
scored AS (
    SELECT
        event_date,
        region,
        event_count,
        active_users,
        purchase_users,
        purchase_revenue,
        LAG(purchase_revenue, 1, 0) OVER (
            PARTITION BY region
            ORDER BY event_date
        ) AS previous_revenue,
        SUM(purchase_revenue) OVER (
            PARTITION BY region
            ORDER BY event_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_revenue,
        RANK() OVER (
            PARTITION BY event_date
            ORDER BY purchase_revenue DESC
        ) AS revenue_rank
    FROM daily_region
)
SELECT
    event_date,
    region,
    event_count,
    active_users,
    purchase_users,
    purchase_revenue,
    previous_revenue,
    cumulative_revenue,
    revenue_rank
FROM scored
ORDER BY event_date, revenue_rank, region
""", title="Daily-region activity, trend, and rank");

<a id="unit-u.aac2d1fb466b4c2fae30524d972574f3"></a>

**Күтілетін нәтиже:** сұрау **24 жол** қайтарады: үш күн × сегіз аймақ. Агрегаттау күн-аймақ түйіршіктілігін орнатады; терезе кезеңі барлық 24 жолды сақтайды.

Уақыт бойымен терезе бағандарын оқу үшін `region_02` пайдаланыңыз:

- `2020-03-01` күнінде `purchase_revenue` — 713,831.18, `previous_revenue` — 0.00 және `cumulative_revenue` — 713,831.18.
- `2020-03-02` күнінде түсім 1,321,173.79 дейін өседі. `previous_revenue` — 713,831.18, жинақталған түсім 2,035,004.97 болады және сол күні аймақ бірінші орын алады.
- `2020-03-03` күнінде `previous_revenue` — 1,321,173.79, жинақталған түсім 2,849,443.38 болады және аймақ сол күні тағы бірінші орын алады.

`active_users` және `purchase_users` әр күн-аймақ тобы ішінде бірегей. Бұл сандарды аймақтар немесе күндер арасында қосып, жиынтықты жаһандық бірегей пайдаланушылар деп түсінбеңіз; сол пайдаланушы бірнеше топта болуы мүмкін.

| Сұрау кезеңі | Нәтиже түйіршіктілігі | Бақыланған жолдар |
|---|---|---:|
| `filtered` | Әр таңдалған оқиғаға бір жол | Үш күннің егжей-тегжейлі жолдары |
| `daily_region` | Әр күн мен аймаққа бір жол | 24 |
| `scored` және соңғы нәтиже | Әр күн мен аймаққа бір жол | 24 |

<a id="unit-u.59eca8cc1e5d413181dd7ce996f62a8f"></a>

<a id="10-apply-a-lambda-expression-to-each-array-element"></a>

## 10. Әр ARRAY элементіне Lambda өрнегін қолдану

Doris Lambda өрнектері көбіне жоғары ретті ARRAY функцияларының ішінде болады. `ARRAY_MAP` әр элементті түрлендіреді, ал `ARRAY_FILTER` шартқа сай элементтерді қалдырады. Бұл шағын литерал әрекетті оқшаулайды; ол `events_modelled` схемасын өзгертпейді.


In [ ]:
lab.sql("""
SELECT
    ARRAY_MAP(x -> x * 2, [1, 2, 3, 4]) AS doubled,
    ARRAY_FILTER(x -> x >= 3, [1, 2, 3, 4]) AS kept
""", title="Lambda expressions in ARRAY functions");


<a id="unit-u.df338803c5af4a8b92c6bac1c444c316"></a>

**Күтілетін нәтиже:** `doubled` — `[2, 4, 6, 8]`, ал `kept` — `[3, 4]`. Lambda өрнегі кіріктірілген жоғары ретті функцияның аргументі; ол бөлек орналастырылған пайдаланушы функциясы (UDF) емес.


<a id="unit-u.61e3a4c3ab7549568471dc3d9550f923"></a>

<a id="11-assemble-a-business-query-from-bounded-choices"></a>

## 11. Шектелген таңдаулардан бизнес сұрауын жинау

Есеп түйіршіктілігін, оқиға сүзгісін және метриканы таңдаңыз. Ноутбук көрінетін Doris SQL құрып, орындайды, сондықтан бос редактордан толық оператор жазбай-ақ әр таңдауды өрнегімен байланыстыра аласыз.

Бұл құрастырғыштағы әр таңдалатын амал Doris кіріктірілген функциясын пайдаланады: `TO_DATE` және `DATE_TRUNC` есеп түйіршіктілігін анықтайды, ал `COUNT`, `COUNT(DISTINCT ...)` және `SUM` метрикаларды есептейді. Бұл функциялар Doris ішінде бар және сыртқы іске асыру не тіркеу қажет етпейді.

Әдепкі таңдаулар күндік сатып алу түсімін сұрайды. Әр жолы бір таңдауды өзгертіп, қай тармақ немесе өрнек өзгеретінін және шығыс түйіршіктілігі өзгеретінін бақылаңыз.


In [ ]:
lab.guided_analysis_builder();


<a id="unit-u.38b038f002cc463d9cf8de012ab26b7d"></a>

**Күтілетін нәтиже:** әдепкі Day, Purchases only және Revenue таңдауларымен алдыңғы нәтиже — 9,933,097.98 мәнімен `2020-03-02`, одан кейін 6,246,556.39 мәнімен `2020-03-03` және 5,670,241.29 мәнімен `2020-03-01`. Week немесе Month таңдау топтау түйіршіктілігін өзгертеді; басқа метрика таңдау агрегат өрнегін өзгертеді.

Жасалған сұрау түгел кіріктірілген функциялардан құралған. 3-модульдің S3 функциясы да кіріктірілген; ол `FROM` үшін уақытша қатынас қайтаратындықтан TVF санатына жатады. Бұл зертханалық жұмыс қашықтағы жиынды қайта оқымайды, өйткені толық объектілік сақтау жүктеуін қайталамай функция санатының әрекетін анықтауға болады.

Әр таңдау үшін кіріс ауқымы [2020-03-01, 2020-03-09) болып қалады. Purchases only және Revenue кезінде Day үш жол, Week екі жол (апта басталуы 2020-02-24 және 2020-03-02), ал Month бір жол береді. Ақпан аптасының белгісі 1 наурызды дүйсенбіден басталатын аптасына топтайды; ақпан оқиғаларын қоспайды. Month нәтижесі тұтас айды емес, осы ауқымды қамтиды.


<a id="unit-u.65afa53175c94f4d98b212e70fc69570"></a>

<a id="12-recognize-when-a-reusable-user-defined-function-is-justified"></a>

## 12. Қайта пайдаланылатын пайдаланушы функциясы қашан негізді екенін тану

Doris кіріктірілген функциялары, айқын SQL өрнектері, Lambda өрнектері және SQL Alias Function көптеген талдамалық талапты қамтиды. Қажетті сала логикасы жоқ болып, сұраулар арасында қайта қолданылуы тиіс және қаптау, орналастыру, рұқсаттар мен жаңарту күтімін ақтаған жағдайда пайдаланушы функциясы тобы қолайлы.

| Кеңейту | Жол әрекеті | Талап мысалы |
|---|---|---|
| Пайдаланушы анықтайтын функция (UDF) | Бір кіріс жолы бір мән береді | Әр тұтынушы оқиғасына меншікті тәуекел ұпайын қолдану |
| Пайдаланушы анықтайтын агрегат функциясы (UDAF) | Бірнеше жол әр топқа бір мән беру үшін күй ұстап, біріктіреді | Doris бермейтін салаға тән топталған метриканы есептеу |
| Пайдаланушы анықтайтын терезелік функция (UDWF) | Терезе әр сақталған нәтиже жолына мән береді | Арнайы терезе күйін және қалпына келтіру ережелерін қолдану |
| Пайдаланушы анықтайтын Table Function (UDTF) | Бір кіріс жолы нөлден көпке дейін жол береді | `LATERAL VIEW` арқылы меншікті кодталған мазмұнды талдап, жаю |

Java скалярлық UDF мына өмірлік циклмен жүреді:

```text
Implement an evaluate method
        ↓
Package the implementation as a JAR
        ↓
Make the JAR available to Doris nodes
        ↓
Register its SQL signature with CREATE FUNCTION
        ↓
Call it from SELECT like another scalar function
```

Келесі мысал — орындалатын ұяшық емес, тіркеу үлгісі. Оған аталған Java класын қамтитын нақты JAR және әкімшілік рұқсаттар қажет:

```sql
CREATE FUNCTION customer_risk_score(BIGINT, DECIMAL(12, 2))
RETURNS DOUBLE
PROPERTIES (
    "file" = "https://example.org/functions/customer-risk.jar",
    "symbol" = "com.example.analytics.CustomerRiskScore",
    "type" = "JAVA_UDF",
    "always_nullable" = "true",
    "volatility" = "immutable"
);

SELECT
    event_id,
    customer_risk_score(user_id, revenue) AS risk_score
FROM events_modelled;
```

`CREATE FUNCTION` SQL атауын, кіріс және қайтару типтерін, іске асыру файлын және класын тіркейді; тәуекел алгоритмін қамтымайды. Backend (BE) сұрау орындалғанда іске асыруды шақырады. Функцияны тек сол кірістер әрдайым сол шығысты бергенде ғана `immutable` деп белгілеңіз.

**Күтілетін түсінік:** бұл зертханалық жұмыстағы орындалатын талдау кіріктірілген функцияларды пайдаланады. UDF, UDAF, UDWF немесе UDTF тек сәйкес жол әрекеті де, қайта пайдаланылатын сала логикасы да қажет болғанда таңдаңыз.

<a id="unit-u.82b1a514eaa549758e405b83f1f59c20"></a>

<a id="13-practice-which-region-grew-most-from-one-day-to-the-next"></a>

## 13. Жаттығу: бір күннен келесісіне қай аймақ ең көп өсті?

9-бөлімде `region_02` 2020-03-02 күнінде ең жоғары сатып алу түсіміне, ал `region_06` екінші ең жоғары түсімге ие болды. Ештеңе орындамас бұрын болжаңыз: 2020-03-01 күнінен 2020-03-02 күніне ең көп өскен де сол екі аймақ па?

Содан кейін 9-бөлімдегідей сұрауды кезеңдермен жазыңыз:

1. 2020-03-01 мен 2020-03-03 аралығындағы әр күнде әр аймақтың сатып алу түсімі бар CTE;
2. аймақ бойынша бөлінген `LAG` арқылы әр жолға сол аймақтың алдыңғы күнінен өзгерісті қосатын CTE;
3. 2020-03-02 күнін қалдырып, сегіз аймаққа сол өзгеріс бойынша `RANK` арқылы орын беретін соңғы `SELECT`.

In [ ]:
# Your code here

<a id="unit-u.c04d11bf960840358ce675d9258b159b"></a>

Келесі ұяшық — үлгі шешім; ол жиналған. Салыстыру үшін оны ашыңыз немесе орындаңыз.

In [ ]:
growth = lab.sql("""
WITH daily_region AS (
    SELECT TO_DATE(event_time) AS event_date,
           region,
           SUM(CASE WHEN event_type = 'purchase' THEN revenue ELSE 0 END) AS purchase_revenue
    FROM events_modelled
    WHERE event_time >= '2020-03-01 00:00:00'
      AND event_time <  '2020-03-04 00:00:00'
    GROUP BY TO_DATE(event_time), region
),
changes AS (
    SELECT event_date,
           region,
           purchase_revenue,
           purchase_revenue - LAG(purchase_revenue) OVER (
               PARTITION BY region ORDER BY event_date
           ) AS change_from_previous_day
    FROM daily_region
)
SELECT event_date,
       region,
       purchase_revenue,
       change_from_previous_day,
       RANK() OVER (PARTITION BY event_date ORDER BY change_from_previous_day DESC) AS growth_rank
FROM changes
WHERE event_date = '2020-03-02'
ORDER BY growth_rank, region
""", title="Growth from 2020-03-01 to 2020-03-02 by region")

assert growth.iloc[0]["region"] == "region_02"
assert len(growth) == 8;

<a id="unit-u.c6da9ebffb4b4d55813bbc3de1f4ded2"></a>

**Күтілетін нәтиже**

Әр аймаққа біреуден сегіз жол. `region_02` ең көп өсті: түсімі **607,342.61** артып, 1,321,173.79 болды. 2020-03-02 күнінде 1,297,553.26 түсіммен екінші тұрған `region_06` өсу бойынша тек **бесінші**, +548,800.32, өйткені 2020-03-01 күнінде ол көбірек сатқан; өсу бойынша екінші — +594,478.09 мәнімен `region_03`.

Кезеңдердің реті маңызды. `LAG` күн сүзгісінен бұрын аймақ бойынша бөлінген күн-аймақ жолдарында орындалуы тиіс: алдымен 2020-03-02 күнін ғана сүзу әр аймаққа бір жол қалдырып, алдыңғы күнді жояр еді, сондықтан әр өзгеріс `NULL` болар еді. Содан кейін соңғы `RANK` сол бір күн ішіндегі аймақтарды салыстырады.

<a id="unit-u.12c15ab978e94c5296af928c9ae62ef1"></a>

<a id="stop-the-doris-sandbox"></a>

### Doris сынақ ортасын тоқтату

CPU мен жадты босату үшін осы қосымша ұяшықты орындаңыз. 5-зертханалық жұмыс `events_modelled` оқиды, бірақ ауыстырмайды; Docker атаулы томдары дерекқор мен кестені сақтайды.


In [ ]:
lab.shell(r"""
set -euo pipefail

docker stop doris-rta
docker inspect --format 'container={{.State.Status}}' doris-rta
""", title="Stop the Doris sandbox");


<a id="unit-u.8a8d346111c649038a1bd1ecbfb39e65"></a>

**Күтілетін нәтиже:** Docker `container=exited` деп хабарлайды.

<a id="restart-the-doris-sandbox"></a>

### Doris сынақ ортасын қайта іске қосу

Басқа модульге өтпес бұрын осы ұяшықты орындаңыз. Ол бар контейнерді іске қосып, FE түйініне қайта қосылады және сақталған сұрауға дайын модельді тексереді.

Бұл қайта іске қосу ұяшығы идемпотентті: оны Docker Desktop пен контейнер тоқтаған, іске қосылып жатқан немесе жұмыс істеп тұрған кезде орындауға болады. macOS жүйесінде қажет болса Docker Desktop ашылады; Linux жүйесінде ұяшықты орындамас бұрын Docker Engine іске қосыңыз.


In [ ]:
lab.start_container()

lab.connect()
lab.execute("USE doris_course")
lab.sql("SELECT COUNT(*) AS event_rows FROM events_modelled", title="Recovered analytical input");


<a id="unit-u.49976663c38047c0b4c2a5bc9b68201f"></a>

**Күтілетін нәтиже:** контейнер `healthy` күйіне оралады және `event_rows` мәні 10,158,080.

<a id="lab-complete"></a>

## Зертханалық жұмыс аяқталды

Функцияларды кіріс, шығыс, жол саны әрекеті және тәуелділік бойынша жіктедіңіз; орындалатын мысалдармен Table Function мен TVF ажыраттыңыз; жеке мәндерді түрлендірдіңіз; егжей-тегжейлі жолдарды топталған метрикаларға азайттыңыз; `WHERE` пен `HAVING` ажыраттыңыз; `ANY_VALUE` тек жарамды шектеумен қолдандыңыз; аралық нәтижелерге CTE арқылы атау бердіңіз; терезелік функциялардың нәтиже жолдарын сақтайтынын байқадыңыз; 24 жолдық күн-аймақ талдауын құрдыңыз; ARRAY мәндеріне Lambda өрнектерін қолдандыңыз; басқарылатын талдамалық сұрау жинадыңыз; және қолдау көрсетілмейтін қайта пайдаланылатын логикаға тиісті пайдаланушы функциясы тобын таңдадыңыз.

Ресми анықтамалар: [STATE](https://doris.apache.org/docs/4.x/sql-manual/sql-functions/combinators/state/) · [EXPLODE](https://doris.apache.org/docs/4.x/sql-manual/sql-functions/table-functions/explode/) · [NUMBERS](https://doris.apache.org/docs/4.x/sql-manual/sql-functions/table-valued-functions/numbers/) · [AI функциялары](https://doris.apache.org/docs/4.x/sql-manual/sql-functions/ai-functions/overview/) · [Common Table Expressions](https://doris.apache.org/docs/4.x/query-data/cte/) · [Терезелік функциялар](https://doris.apache.org/docs/4.x/query-data/window-function/) · [ANY_VALUE](https://doris.apache.org/docs/4.x/sql-manual/sql-functions/aggregate-functions/any-value/) · [ARRAY_MAP](https://doris.apache.org/docs/4.x/sql-manual/sql-functions/scalar-functions/array-functions/array-map/) · [Java UDF, UDAF, UDWF және UDTF](https://doris.apache.org/docs/4.x/query-data/udf/java-user-defined-function/) · [CREATE FUNCTION](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/function/CREATE-FUNCTION/)

<a id="unit-u.e071642d70a74bd6b410fb04a4ca1042"></a>

<a id="next"></a>

## Келесі қадам

[5-тест: Apache Doris жүйесінде деректерді талдау](quiz05_analytical_query_semantics.ipynb) ашыңыз, содан кейін [6-модуль: Apache Doris жүйесінде деректерді біріктіру](../module06-joining/course.md) бөліміне өтіңіз.
